# Version 14: Dilated ResNet + BiGRU (The "SOTA" Attempt)

**Goal**: Outperform all previous versions (Target > 0.60)

**Architecture Upgrade:**
- **Dilated Residual CNN (ResNet-1D)**: Replaces simple Multi-Scale CNN. Uses dilations [1, 2, 4, 8] to capture wide local context (~30 residues) *before* the RNN.
- **BiGRU**: 2 Layers (Global context). Reduced from 3 to balance parameter budget.
- **Self-Attention**: Retained for global reasoning.
- **Features**: Physico-chemical properties integrated.


In [ ]:
!pip install -q trackio

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.amp import autocast
from torch.cuda.amp import GradScaler
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

torch.backends.cudnn.benchmark = True

class Config:
    seed = 42
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Architecture (ResNet Hybrid)
    embed_dim = 128
    physico_dim = 4
    cnn_channels = 64
    hidden_dim = 384
    num_layers = 2     # BiGRU layers
    num_heads = 4
    dropout = 0.35
    
    # Training
    batch_size = 32
    grad_accum = 2
    lr = 6e-4          # Slightly higher than V10 due to ResNet depth
    weight_decay = 1e-4
    epochs = 45
    grad_clip = 1.0
    label_smoothing = 0.1
    use_amp = True
    
    val_split = 0.1
    train_path = 'train.csv'
    test_path = 'test.csv'
    model_save_path = 'best_model_v14.pth'
    submission_path = 'submission_v14.csv'
    
    trackio_project = '25-t3-nppe2'
    trackio_space_id = 'manyakaistha/dlgenai-nppe'
    trackio_group = 'v14_resnet'
    
    max_class_weight = 3.0

cfg = Config()
print(f'Device: {cfg.device}')

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

In [ ]:
class Mish(nn.Module):
    def forward(self, x): return x * torch.tanh(nn.functional.softplus(x))

class ResidualBlock(nn.Module):
    """Dilated Residual Block for 1D Sequence"""
    def __init__(self, channels, kernel_size, dilation, dropout=0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.norm1 = nn.LayerNorm(channels)
        self.act1 = Mish()
        self.dropout1 = nn.Dropout(dropout)
        
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.norm2 = nn.LayerNorm(channels)
        self.act2 = Mish()
        self.dropout2 = nn.Dropout(dropout)
        
    def forward(self, x):
        # x: [B, C, L]
        residual = x
        out = self.conv1(x)
        out = out.transpose(1, 2) # [B, L, C]
        out = self.norm1(out)
        out = self.act1(out)
        out = self.dropout1(out).transpose(1, 2)
        
        out = self.conv2(out)
        out = out.transpose(1, 2) 
        out = self.norm2(out)
        out = self.act2(out)
        out = self.dropout2(out).transpose(1, 2)
        
        return out + residual

class DilatedResNet(nn.Module):
    """Stack of Residual Blocks with increasing dilation"""
    def __init__(self, in_dim, channels, dilations=[1, 2, 4, 8]):
        super().__init__()
        self.stem = nn.Linear(in_dim, channels) # Project to channel dim
        self.blocks = nn.ModuleList([
            ResidualBlock(channels, kernel_size=3, dilation=d) for d in dilations
        ])
    
    def forward(self, x):
        # x: [B, L, D]
        x = self.stem(x).transpose(1, 2) # [B, C, L]
        for block in self.blocks:
            x = block(x)
        return x.transpose(1, 2) # [B, L, C]


In [ ]:
os.environ['HUGGINGFACEHUB_API_TOKEN'] = 'hf_placeholder_token_for_reproducibility'
os.environ['TRACKIO_SPACE_ID'] = cfg.trackio_space_id

def maybe_init_trackio():
    try:
        import trackio
        if cfg.trackio_space_id:
            trackio.init(project=cfg.trackio_project, space_id=cfg.trackio_space_id, group=cfg.trackio_group)
            print('✓ TrackIO initialized')
            return trackio
    except Exception as e:
        print(f'TrackIO: {e}')
    return None

trk = maybe_init_trackio()

# Vocab & Data
AA_PROPS = {'<PAD>': [0, 0, 0, 0], '<UNK>': [0, 0, 0, 0], '*': [0, 0, 0, 0],
    'A': [0.4, 0, 0.45, 0.43], 'R': [-1.0, 1, 0.87, 0.77], 'N': [-0.78, 0, 0.66, 0.39],
    'D': [-0.78, -1, 0.67, 0.20], 'C': [0.56, 0, 0.61, 0.36], 'E': [-0.78, -1, 0.74, 0.23],
    'Q': [-0.78, 0, 0.73, 0.40], 'G': [-0.09, 0, 0.38, 0.43], 'H': [-0.71, 0.5, 0.78, 0.54],
    'I': [1.0, 0, 0.66, 0.43], 'L': [0.84, 0, 0.66, 0.43], 'K': [-0.87, 1, 0.73, 0.70],
    'M': [0.42, 0, 0.75, 0.41], 'F': [0.62, 0, 0.83, 0.39], 'P': [-0.36, 0, 0.58, 0.45],
    'S': [-0.18, 0, 0.53, 0.41], 'T': [-0.16, 0, 0.60, 0.40], 'W': [-0.20, 0, 1.02, 0.42],
    'Y': [-0.29, 0, 0.91, 0.40], 'V': [0.93, 0, 0.59, 0.43]}

PAD_TOKEN = 0
AA_CHARS = ['<PAD>', '<UNK>', '*', 'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
AA_TO_ID = {c: i for i, c in enumerate(AA_CHARS)}
ID_TO_AA = {i: c for c, i in AA_TO_ID.items()}

Q8_CHARS = ['<PAD>'] + sorted(['B', 'C', 'E', 'G', 'H', 'I', 'S', 'T'])
Q8_TO_ID = {c: i for i, c in enumerate(Q8_CHARS)}
ID_TO_Q8 = {i: c for c, i in Q8_TO_ID.items()}

Q3_CHARS = ['<PAD>'] + sorted(['C', 'E', 'H'])
Q3_TO_ID = {c: i for i, c in enumerate(Q3_CHARS)}
ID_TO_Q3 = {i: c for c, i in Q3_TO_ID.items()}

prop_tensor = torch.zeros(len(AA_CHARS), cfg.physico_dim)
for char, idx in AA_TO_ID.items():
    if char in AA_PROPS: prop_tensor[idx] = torch.tensor(AA_PROPS[char])

class ProteinDataset(Dataset):
    def __init__(self, df, mode='train'):
        self.df = df
        self.mode = mode
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = [AA_TO_ID.get(c, AA_TO_ID['<UNK>']) for c in row['seq']]
        item = {'seq': torch.tensor(seq, dtype=torch.long), 'id': row['id'], 'len': len(seq)}
        if self.mode == 'train':
            item['q8'] = torch.tensor([Q8_TO_ID[c] for c in row['sst8']], dtype=torch.long)
            item['q3'] = torch.tensor([Q3_TO_ID[c] for c in row['sst3']], dtype=torch.long)
        return item

def collate_fn(batch):
    batch.sort(key=lambda x: x['len'], reverse=True)
    seqs = pad_sequence([x['seq'] for x in batch], batch_first=True, padding_value=PAD_TOKEN)
    lengths = torch.tensor([x['len'] for x in batch])
    out = {'seq': seqs, 'len': lengths, 'ids': [x['id'] for x in batch]}
    if 'q8' in batch[0]:
        out['q8'] = pad_sequence([x['q8'] for x in batch], batch_first=True, padding_value=PAD_TOKEN)
        out['q3'] = pad_sequence([x['q3'] for x in batch], batch_first=True, padding_value=PAD_TOKEN)
    return out

df_full = pd.read_csv(cfg.train_path)
train_df, val_df = train_test_split(df_full, test_size=cfg.val_split, random_state=cfg.seed)
train_loader = DataLoader(ProteinDataset(train_df), batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn, pin_memory=True, num_workers=2)
val_loader = DataLoader(ProteinDataset(val_df), batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=2)

def compute_class_weights(labels_column, vocab, max_weight=cfg.max_class_weight):
    counts = {c: 0 for c in vocab if c != '<PAD>'}
    for label_str in labels_column:
        for c in label_str: 
            if c in counts: counts[c] += 1
    total = sum(counts.values())
    weights = torch.ones(len(vocab))
    for c, cnt in counts.items():
        if cnt > 0:
            idx = vocab.index(c)
            weights[idx] = min(total / (len(counts) * cnt), max_weight)
    weights[0] = 0.0
    return weights / weights[1:].max()

q8_w = compute_class_weights(train_df['sst8'], Q8_CHARS).to(cfg.device)
q3_w = compute_class_weights(train_df['sst3'], Q3_CHARS).to(cfg.device)

In [ ]:
class EfficientSelfAttention(nn.Module):
    def __init__(self, dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        attn_out, _ = self.attn(x, x, x, key_padding_mask=~mask if mask is not None else None, need_weights=False)
        return self.norm(x + self.dropout(attn_out))

class ResNetBiGRU(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Embed
        self.embedding = nn.Embedding(len(AA_TO_ID), cfg.embed_dim, padding_idx=PAD_TOKEN)
        self.prop_embed = nn.Embedding.from_pretrained(prop_tensor, freeze=False)
        self.prop_proj = nn.Linear(cfg.physico_dim, 16)
        
        # Dilated ResNet (The "Convd" Upgrade)
        input_dim = cfg.embed_dim + 16
        self.resnet = DilatedResNet(input_dim, cfg.cnn_channels, dilations=[1, 2, 4, 8])
        
        # Bi-GRU
        gru_in = cfg.cnn_channels
        self.gru = nn.GRU(gru_in, cfg.hidden_dim, cfg.num_layers, 
                          batch_first=True, bidirectional=True,
                          dropout=cfg.dropout)
        self.gru_norm = nn.LayerNorm(cfg.hidden_dim * 2)
        
        # Residual & Attention
        self.residual_proj = nn.Linear(gru_in, cfg.hidden_dim * 2)
        self.attention = EfficientSelfAttention(cfg.hidden_dim * 2, cfg.num_heads, cfg.dropout)
        
        # Heads
        self.dropout = nn.Dropout(cfg.dropout)
        self.q8_head = nn.Sequential(
            nn.Linear(cfg.hidden_dim * 2, 128), Mish(), nn.Dropout(cfg.dropout),
            nn.Linear(128, len(Q8_TO_ID))
        )
        self.q3_head = nn.Sequential(
            nn.Linear(cfg.hidden_dim * 2, 64), Mish(), nn.Dropout(cfg.dropout),
            nn.Linear(64, len(Q3_TO_ID))
        )
    
    def forward(self, x, lengths, mask):
        # Embed
        embed = self.embedding(x)
        props = self.prop_proj(self.prop_embed(x))
        x_in = torch.cat([embed, props], dim=2)
        
        # ResNet
        resnet_out = self.resnet(x_in)
        
        # GRU with Packing
        lengths_cpu = lengths.cpu()
        packed = pack_padded_sequence(resnet_out, lengths_cpu, batch_first=True, enforce_sorted=True)
        packed_out, _ = self.gru(packed)
        gru_out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=x.size(1))
        
        # Residual (ResNet features -> GRU output)
        res = self.residual_proj(resnet_out)
        gru_out = self.gru_norm(gru_out + res)
        
        # Attention
        attn_out = self.attention(gru_out, mask)
        
        # Heads
        out = self.dropout(attn_out)
        return self.q8_head(out), self.q3_head(out)

model = ResNetBiGRU().to(cfg.device)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
criterion_q8 = nn.CrossEntropyLoss(weight=q8_w, ignore_index=PAD_TOKEN, label_smoothing=cfg.label_smoothing)
criterion_q3 = nn.CrossEntropyLoss(weight=q3_w, ignore_index=PAD_TOKEN, label_smoothing=cfg.label_smoothing)

optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
scaler = GradScaler(enabled=cfg.use_amp)

def harmonic_mean(a, b): return 2 * (a * b) / (a + b + 1e-8)

def calculate_f1(preds, targets):
    preds_flat = preds.argmax(dim=-1).view(-1).cpu().numpy()
    targets_flat = targets.view(-1).cpu().numpy()
    mask = targets_flat != PAD_TOKEN
    return f1_score(targets_flat[mask], preds_flat[mask], average='macro', zero_division=0)

def train_epoch(model, loader, optimizer, scheduler, scaler, device, epoch):
    model.train()
    total_loss, total_q8_f1, total_q3_f1, count = 0, 0, 0, 0
    optimizer.zero_grad()
    
    pbar = tqdm(loader, desc=f'Epoch {epoch+1} Train')
    for step, batch in enumerate(pbar):
        seq = batch['seq'].to(device, non_blocking=True)
        lengths = batch['len'].to(device, non_blocking=True)
        mask = (seq != PAD_TOKEN)
        q8_tgt = batch['q8'].to(device, non_blocking=True)
        q3_tgt = batch['q3'].to(device, non_blocking=True)
        
        with autocast(device_type='cuda', enabled=cfg.use_amp):
            q8_logits, q3_logits = model(seq, lengths, mask)
            loss_q8 = criterion_q8(q8_logits.view(-1, len(Q8_TO_ID)), q8_tgt.view(-1))
            loss_q3 = criterion_q3(q3_logits.view(-1, len(Q3_TO_ID)), q3_tgt.view(-1))
            loss = (loss_q8 + loss_q3) / cfg.grad_accum
        
        scaler.scale(loss).backward()
        
        if (step + 1) % cfg.grad_accum == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step(epoch + step / len(loader))
        
        total_loss += (loss_q8.item() + loss_q3.item())
        
        with torch.no_grad():
            f1_8 = calculate_f1(q8_logits, q8_tgt)
            f1_3 = calculate_f1(q3_logits, q3_tgt)
            total_q8_f1 += f1_8
            total_q3_f1 += f1_3
        count += 1
        pbar.set_postfix({'loss': total_loss/count, 'q8': total_q8_f1/count})
    
    return total_loss/count, total_q8_f1/count, total_q3_f1/count, harmonic_mean(total_q8_f1/count, total_q3_f1/count)

def validate(model, loader, device):
    model.eval()
    total_loss = 0
    all_q8_preds, all_q8_targets, all_q3_preds, all_q3_targets = [], [], [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validation'):
            seq = batch['seq'].to(device, non_blocking=True)
            lengths = batch['len'].to(device, non_blocking=True)
            mask = (seq != PAD_TOKEN)
            q8_tgt = batch['q8'].to(device, non_blocking=True)
            q3_tgt = batch['q3'].to(device, non_blocking=True)
            with autocast(device_type='cuda', enabled=cfg.use_amp):
                q8_logits, q3_logits = model(seq, lengths, mask)
                loss_q8 = criterion_q8(q8_logits.view(-1, len(Q8_TO_ID)), q8_tgt.view(-1))
                loss_q3 = criterion_q3(q3_logits.view(-1, len(Q3_TO_ID)), q3_tgt.view(-1))
            total_loss += (loss_q8 + loss_q3).item()
            mask_bool = q8_tgt != PAD_TOKEN
            all_q8_preds.extend(q8_logits.argmax(-1)[mask_bool].cpu().numpy())
            all_q8_targets.extend(q8_tgt[mask_bool].cpu().numpy())
            all_q3_preds.extend(q3_logits.argmax(-1)[mask_bool].cpu().numpy())
            all_q3_targets.extend(q3_tgt[mask_bool].cpu().numpy())
            
    f1_q8 = f1_score(all_q8_targets, all_q8_preds, average='macro', zero_division=0)
    f1_q3 = f1_score(all_q3_targets, all_q3_preds, average='macro', zero_division=0)
    return total_loss/len(loader), f1_q8, f1_q3, harmonic_mean(f1_q8, f1_q3)

best_score = 0
print('Starting V14 ResNet-BiGRU Training...\n')

for epoch in range(cfg.epochs):
    train_loss, train_q8, train_q3, train_harm = train_epoch(model, train_loader, optimizer, scheduler, scaler, cfg.device, epoch)
    val_loss, val_q8, val_q3, val_harm = validate(model, val_loader, cfg.device)
    
    print(f'\nEpoch {epoch+1}/{cfg.epochs}:')
    print(f'  Train Loss: {train_loss:.4f} | Q8: {train_q8:.4f} | Q3: {train_q3:.4f} | Harm: {train_harm:.4f}')
    print(f'  Val   Loss: {val_loss:.4f} | Q8: {val_q8:.4f} | Q3: {val_q3:.4f} | Harm: {val_harm:.4f}')
    
    if val_harm > best_score:
        best_score = val_harm
        torch.save(model.state_dict(), cfg.model_save_path)
        print(f'✓ Saved best model (epoch {epoch+1}, score={val_harm:.4f})')
    
    print('-' * 60)
    if trk:
        trk.log({'val_harm': val_harm, 'val_q8': val_q8, 'val_loss': val_loss})

print(f'\n✓ Done. Best: {best_score:.4f}')

In [ ]:
model.load_state_dict(torch.load(cfg.model_save_path))
model.eval()
test_df = pd.read_csv(cfg.test_path)
test_loader = DataLoader(ProteinDataset(test_df, 'test'), batch_size=cfg.batch_size, collate_fn=collate_fn)
res = []
with torch.no_grad():
    for batch in tqdm(test_loader):
        seq = batch['seq'].to(cfg.device)
        lengths = batch['len'].to(cfg.device)
        mask = (seq != PAD_TOKEN)
        p8, p3 = model(seq, lengths, mask)
        p8 = p8.argmax(-1)
        p3 = p3.argmax(-1)
        for i, L in enumerate(batch['len']):
            s8 = ''.join([ID_TO_Q8[x] for x in p8[i][:L].cpu().numpy()])
            s3 = ''.join([ID_TO_Q3[x] for x in p3[i, :L].cpu().numpy()])
            res.append({'id': batch['ids'][i], 'sst8': s8, 'sst3': s3})
pd.DataFrame(res).to_csv(cfg.submission_path, index=False)
print('✓ Inference done')